# ETL Spotify — Extracción de Datos Comerciales y B2C

Este notebook implementa un pipeline ETL que consume la **API de Spotify** a través de `spotipy`
y genera dos datasets para alimentar dashboards de **Business Intelligence (BI)** y **recomendaciones B2C**.

## Flujo del pipeline

1. **Autenticación** via `Client Credentials` de Spotify.
2. **Extracción** — Por cada género musical:
   - Top 10 artistas (nombre, id, popularidad, seguidores, imagen).
   - Top 3 pistas por artista (nombre, id, popularidad, preview_url, imagen de álbum).
3. **Demografía sintética** — Columnas de edad y sexo simuladas con valores realistas por género.
4. **Exportación** — Dos CSVs listos para consumir desde Tableau / Streamlit / Power BI.

## Salidas generadas

| Archivo | Contenido |
|---|---|
| `data/processed/spotify_business_metrics.csv` | Métricas agregadas por género + demografía sintética (7 filas) |
| `data/processed/spotify_b2c_recommendations.csv` | Top artistas y canciones por género (~210 filas) |

In [ ]:
import os
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import spotipy
from spotipy.exceptions import SpotifyException
from spotipy.oauth2 import SpotifyClientCredentials

warnings.filterwarnings("ignore")
print("Librerias cargadas OK")

In [ ]:
# ============================================================
# CONFIGURACION
# ============================================================

# Credenciales de Spotify API (reemplazar con valores reales)
CLIENT_ID = "33b68ecc2eda420092e06076d15cbfbe"
CLIENT_SECRET = "ad84339bd85a433e9b6f9b91cd52a90e"

# Lista base de generos musicales a consultar
GENRES = ["pop", "rock", "jazz", "hip-hop", "classical", "electronic", "reggaeton"]

# Limites de extraccion
TOP_N_ARTISTS = 10
TOP_N_TRACKS = 3

# Directorio de salida (crea data/ relativo a Data/)
DATA_DIR = Path("../data/processed")
DATA_DIR.mkdir(parents=True, exist_ok=True)
print(f"Directorio de salida: {DATA_DIR.resolve()}")

# Base de datos demograficos sinteticos por genero
DEMOGRAPHIC_BASE = {
    "pop":        {"edad": 25, "hombres": 38, "mujeres": 57, "ganancia": 950_000},
    "rock":       {"edad": 34, "hombres": 62, "mujeres": 33, "ganancia": 720_000},
    "jazz":       {"edad": 47, "hombres": 52, "mujeres": 43, "ganancia": 310_000},
    "hip-hop":    {"edad": 23, "hombres": 68, "mujeres": 28, "ganancia": 880_000},
    "classical":  {"edad": 52, "hombres": 44, "mujeres": 51, "ganancia": 280_000},
    "electronic": {"edad": 28, "hombres": 56, "mujeres": 39, "ganancia": 540_000},
    "reggaeton":  {"edad": 22, "hombres": 43, "mujeres": 52, "ganancia": 780_000},
}
print("Configuracion cargada OK")

In [ ]:
def setup_spotify() -> spotipy.Spotify:
    """Inicializa y devuelve un cliente autenticado de Spotify."""
    try:
        auth_manager = SpotifyClientCredentials(
            client_id=CLIENT_ID,
            client_secret=CLIENT_SECRET,
        )
        sp = spotipy.Spotify(auth_manager=auth_manager)
        # Prueba rapida de conectividad
        sp.search(q="test", type="artist", limit=1)
        print("Autenticacion exitosa contra la API de Spotify")
        return sp
    except SpotifyException as e:
        print(f"Error de autenticacion: {e}")
        raise
    except Exception as e:
        print(f"Error inesperado al conectar con Spotify: {e}")
        raise

In [ ]:
def _api_call_with_retry(
    func: callable,
    *args,
    max_retries: int = 5,
    **kwargs,
) -> dict | None:
    """Ejecuta una llamada a la API de Spotify con reintento
    exponencial ante errores 429 (Rate Limit).

    Parametros
    ----------
    func : callable
        Metodo de spotipy.Spotify a invocar.
    max_retries : int
        Numero maximo de reintentos.

    Retorna
    -------
    dict | None
        Respuesta de la API, o None si falla tras todos los reintentos.
    """
    for attempt in range(1, max_retries + 1):
        try:
            return func(*args, **kwargs)
        except SpotifyException as e:
            if e.http_status == 429:
                retry_after = int(
                    getattr(e, "headers", {}).get("Retry-After", 2**attempt)
                )
                print(
                    f"  Rate limit (429). Reintentando en {retry_after}s "
                    f"(intento {attempt}/{max_retries})"
                )
                time.sleep(retry_after)
            else:
                print(f"  Error Spotify ({e.http_status}): {e.msg}")
                return None
        except Exception as e:
            print(f"  Error inesperado: {e}")
            return None
    print(f"  Se agotaron los reintentos ({max_retries})")
    return None

In [ ]:
def search_artists_by_genre(
    sp: spotipy.Spotify,
    genre: str,
    limit: int = 10,
) -> list[dict]:
    """Busca los artistas mas relevantes para un genero dado.

    Primero intenta genre:\"genre\", si da 0 resultados hace un
    fallback a busqueda por nombre del genero.

    El search basico no incluye popularity ni followers, por lo que
    se hace una llamada adicional por artista al endpoint de detalle.
    """
    try:
        # Intento 1: busqueda por genero exacto
        results = _api_call_with_retry(
            sp.search, q=f'genre:"{genre}"', type="artist", limit=limit
        )
        artists = []
        if results:
            artists = results.get("artists", {}).get("items", [])

        # Intento 2: fallback por nombre del genero
        if not artists:
            print(f"  Sin resultados con genre:, probando busqueda por nombre...")
            results = _api_call_with_retry(
                sp.search, q=genre, type="artist", limit=limit
            )
            if results:
                artists = results.get("artists", {}).get("items", [])

        parsed = []
        for a in artists:
            full = _api_call_with_retry(sp.artist, a["id"])
            if full is None:
                continue
            parsed.append({
                "id": a["id"],
                "name": a["name"],
                "popularity": full.get("popularity") or 0,
                "followers": (full.get("followers") or {}).get("total", 0),
                "image_url": a["images"][0]["url"] if a["images"] else "",
                "genres": full.get("genres", []),
            })
        print(f"  {genre}: {len(parsed)} artistas encontrados")
        return parsed
    except Exception as e:
        print(f"  Error buscando artistas para {genre}: {e}")
        return []


In [ ]:
def get_top_tracks(
    sp: spotipy.Spotify,
    artist_name: str,
    limit: int = 3,
) -> list[dict]:
    """Obtiene las pistas mas populares de un artista via search.

    artist_top_tracks requiere auth de usuario (da 403 con Client Credentials),
    por lo que se usa sp.search con filtro artist:.

    Nota: El endpoint search de tracks no incluye el campo popularity,
    por lo que se usa un valor default 0 (el orden depende del algoritmo
    de relevancia de Spotify).
    """
    try:
        results = _api_call_with_retry(
            sp.search, q=f'artist:"{artist_name}"', type="track", limit=limit,
        )
        if results is None:
            return []

        tracks = results.get("tracks", {}).get("items", [])
        # Filtrar tracks cuyo artista principal coincida exactamente
        top = [
            t for t in tracks
            if any(a["name"].lower() == artist_name.lower()
                   for a in t.get("artists", []))
        ][:limit]

        parsed = []
        for t in top:
            parsed.append({
                "id": t["id"],
                "name": t["name"],
                "popularity": t.get("popularity", 0),
                "preview_url": t.get("preview_url") or "",
                "album_image": (
                    t["album"]["images"][0]["url"]
                    if t["album"].get("images")
                    else ""
                ),
            })
        return parsed
    except Exception as e:
        print(f"    Error obteniendo tracks de {artist_name}: {e}")
        return []


In [ ]:
def extract_b2c_data(
    sp: spotipy.Spotify,
    genres: list[str],
) -> pd.DataFrame:
    """Construye el dataset plano de recomendaciones B2C.

    Para cada genero extrae top N artistas y top M pistas por artista,
    consolidando todo en un DataFrame fila por cancion.
    """
    rows = []

    for genre in genres:
        print(f"\nProcesando genero: {genre}")
        artists = search_artists_by_genre(sp, genre, limit=TOP_N_ARTISTS)

        for artist in artists:
            tracks = get_top_tracks(sp, artist["name"], limit=TOP_N_TRACKS)
            for track in tracks:
                rows.append({
                    "genero": genre,
                    "artista_nombre": artist["name"],
                    "artista_id": artist["id"],
                    "popularidad_artista": artist["popularity"],
                    "seguidores_artista": artist["followers"],
                    "imagen_artista": artist["image_url"],
                    "cancion_nombre": track["name"],
                    "cancion_id": track["id"],
                    "popularidad_cancion": track["popularity"],
                    "url_preview": track["preview_url"],
                    "imagen_album": track["album_image"],
                })
            print(f"  {artist['name']}: {len(tracks)} pistas")

    df = pd.DataFrame(rows)
    print(f"\nB2C: {len(df)} registros generados")
    return df


In [ ]:
def generate_demographics(genre: str, seed: int | None = None) -> dict:
    """Genera datos demograficos sinteticos realistas para un genero.

    Aplica ruido gaussiano sobre los valores base para simular
    variabilidad natural en cada extraccion.

    Parametros
    ----------
    genre : str
        Nombre del genero musical.
    seed : int | None
        Semilla opcional para reproducibilidad.

    Retorna
    -------
    dict
        Edad promedio, porcentajes de genero y ganancia potencial.
    """
    rng = np.random.default_rng(seed)
    base = DEMOGRAPHIC_BASE[genre]

    edad = max(14, rng.normal(loc=base["edad"], scale=2.0))
    hombres = rng.normal(loc=base["hombres"], scale=3.0)
    mujeres = rng.normal(loc=base["mujeres"], scale=3.0)

    # Asegurar que los porcentajes sumen ~100
    total = hombres + mujeres
    if total > 0:
        hombres = max(0, hombres / total * 100)
        mujeres = max(0, mujeres / total * 100)

    ganancia = max(100_000, rng.normal(loc=base["ganancia"], scale=50_000))

    return {
        "edad_promedio_oyente": round(edad, 1),
        "porcentaje_hombres": round(hombres, 1),
        "porcentaje_mujeres": round(mujeres, 1),
        "potencial_ganancia_usd": int(ganancia),
    }

In [ ]:
def extract_business_metrics(
    sp: spotipy.Spotify,
    genres: list[str],
) -> pd.DataFrame:
    """Construye el dataset de metricas de negocio por genero.

    Combina datos agregados reales de la API (popularidad promedio,
    seguidores totales) con datos demograficos sinteticos.

    Parametros
    ----------
    sp : spotipy.Spotify
        Cliente autenticado.
    genres : list[str]
        Lista de generos a procesar.

    Retorna
    -------
    pd.DataFrame
        Columnas: genero, popularidad_promedio, seguidores_totales,
        edad_promedio_oyente, porcentaje_hombres, porcentaje_mujeres,
        potencial_ganancia_usd.
    """
    rows = []

    for genre in genres:
        print(f"\nProcesando metricas para: {genre}")
        artists = search_artists_by_genre(sp, genre, limit=TOP_N_ARTISTS)

        if not artists:
            print(f"  Sin datos de API para {genre}, usando defaults")
            pop_prom = 50.0
            seg_total = 100_000
        else:
            pop_prom = float(np.mean([a["popularity"] for a in artists]))
            seg_total = sum(a["followers"] for a in artists)

        demo = generate_demographics(genre)

        rows.append({
            "genero": genre,
            "popularidad_promedio": round(pop_prom, 1),
            "seguidores_totales": seg_total,
            **demo,
        })

    df = pd.DataFrame(rows)
    print(f"\nBusiness Metrics: {len(df)} generos procesados")
    return df

In [ ]:
def export_datasets(
    df_business: pd.DataFrame,
    df_b2c: pd.DataFrame,
    data_dir: Path = DATA_DIR,
) -> None:
    """Exporta ambos DataFrames a archivos CSV.

    Parametros
    ----------
    df_business : pd.DataFrame
        Dataset de metricas de negocio.
    df_b2c : pd.DataFrame
        Dataset de recomendaciones B2C.
    data_dir : Path
        Directorio de salida.
    """
    business_path = data_dir / "spotify_business_metrics.csv"
    b2c_path = data_dir / "spotify_b2c_recommendations.csv"

    df_business.to_csv(business_path, index=False)
    print(f"Business Metrics exportado: {business_path.resolve()}")

    df_b2c.to_csv(b2c_path, index=False)
    print(f"B2C Recommendations exportado: {b2c_path.resolve()}")

    print(f"\nResumen:")
    print(f"  Business Metrics: {len(df_business)} filas, "
          f"{list(df_business.columns)}")
    print(f"  B2C Recommendations: {len(df_b2c)} filas, "
          f"{list(df_b2c.columns)}")

In [ ]:
def main() -> None:
    """Orquestador principal del pipeline ETL."""
    print("=" * 55)
    print("  ETL SPOTIFY - INICIANDO PIPELINE")
    print("=" * 55)

    # 1. Autenticacion
    print("\n[1/4] Autenticando...")
    try:
        sp = setup_spotify()
    except Exception:
        print("No se pudo autenticar. Revisa CLIENT_ID y CLIENT_SECRET.")
        return

    # 2. Extraccion B2C
    print("\n[2/4] Extrayendo datos B2C...")
    try:
        df_b2c = extract_b2c_data(sp, GENRES)
    except Exception as e:
        print(f"Error en extraccion B2C: {e}")
        df_b2c = pd.DataFrame()

    # 3. Extraccion Business Metrics
    print("\n[3/4] Extrayendo metricas de negocio...")
    try:
        df_business = extract_business_metrics(sp, GENRES)
    except Exception as e:
        print(f"Error en extraccion Business: {e}")
        df_business = pd.DataFrame()

    # 4. Exportacion
    print("\n[4/4] Exportando datasets...")
    try:
        export_datasets(df_business, df_b2c)
    except Exception as e:
        print(f"Error en exportacion: {e}")

    print("\n" + "=" * 55)
    print("  PIPELINE ETL COMPLETADO")
    print("=" * 55)

In [ ]:
if __name__ == "__main__":
    main()